# 検索拡張生成（RAG）システム、LangChain、ChromaDB 入門

このノートブックでは、情報を検索して応答を生成し、特定の情報に基づいて LLM を効果的にグラウンディング（根拠付け）する質問応答（Q&A）システムの構築手順を説明します。事前トレーニング済みの LLM や、ファインチューニングされた LLM でさえも、トレーニング データセットに含まれていなかった特定の情報や非公開データを理解させたい場合、それ単体では十分でないことがよくあります。

このラボで行うこと:
* 検索拡張生成システムのさまざまなコンポーネントについて学ぶ
* シンプルな検索拡張生成（RAG）システムを構築する
* LangChain と ChromaDB を使用してプロセスを簡素化・スケールアップする


### セットアップ


In [ ]:
import pandas as pd
import scipy
from google import genai
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.chains import RetrievalQA
import wikipediaapi
from langchain_core.documents import Document
from langchain_classic.memory import ConversationBufferMemory
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
EMBEDDING_MODEL = "text-embedding-005"
GENERATIVE_MODEL = "gemini-3.5-flash"

### シンプルな検索拡張生成システムの構築


このトイ エグザンプル（簡単な例）では、市販の既製 LLM が知るはずのない情報に基づいて LLM をグラウンディングします。たとえば、2 匹のペットの世話をするハウスシッターのために残された指示などです。


In [ ]:
# LLM をグラウンディングさせたい情報のリスト。
information = [
    "Estrella is a dog",
    "Finnegan is a cat",
    "Finnegan gets fed five times daily. Estrella gets fed three times daily.",
    "Estrella usually goes on one long walk per day, but needs to go outside every 4-6 hours",
    "Please play with Finnegan for 30 minutes each day. His favorite toy is the fake mouse!",
]

information_df = pd.DataFrame({"text": information})
information_df.head()


ほとんどの検索拡張生成システムの中核にあるのはベクトル データベースです。ベクトル データベースは、情報の埋め込み表現を保存します。

テキストの埋め込み表現となる列を information DataFrame に追加しましょう。[Text Embeddings API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/embeddings/get-text-embeddings) を使用します。


In [ ]:
client = genai.Client(vertexai=True, location="global")

In [ ]:
information_df["vector"] = [
    x.values
    for x in client.models.embed_content(
        model=EMBEDDING_MODEL, contents=information
    ).embeddings
]
information_df.head()

検索システムには、指定されたクエリに回答するために最も関連性の高い情報を見つける方法が必要です。これは最近傍（セマンティック類似度）探索によって行われます。クエリ（テキスト）入力を受け取り、information 内の各テキストに対する距離指標を返す関数を定義しましょう。以下の処理が必要です:
* 情報に使用したのと同じ埋め込みモデルでクエリを埋め込む
* クエリ ベクトルと各情報ベクトルとの間の距離指標を計算する（利用可能な多くの類似度尺度の 1 つであるコサイン類似度を使用）
* クエリ ベクトルと各情報ベクトルとの間の距離指標のリストを返す


### 演習
下の関数を完成させ、`information_df` を反復処理してクエリの埋め込みと各情報の埋め込みとの間の類似度スコアを計算するようにしてください。


In [ ]:
def embed_and_compute_distances(query: str):
    # クエリ文字列のベクトルを取得
    query_embedding = (
        client.models.embed_content(model=EMBEDDING_MODEL, contents=query)
        .embeddings[0]
        .values
    )  # クエリの埋め込み

    distances = []

    # TODO: クエリ ベクトルとすべての情報ベクトルとの間の距離を計算します
    # 返されるリストの各要素は、キー 'information' と 'distance' を持つ辞書である必要があります。距離の計算には
    # scipy.spatial.distance.cosine(v1, v2) を使用してください

    return distances


例を使用してこの関数をテストしてみましょう。


In [ ]:
embed_and_compute_distances(query="What type of animal is Estrella?")

"What type of animal is Estrella?"（Estrella はどんな動物ですか？）のベクトルに最も近い（つまり最も類似している）ベクトルが、"Estrella is a dog"（Estrella は犬です）のベクトルであることに注目してください。これは、検索拡張システムを支える核心的な前提を浮き彫りにしています。すなわち、質問に答えるのに関連する情報は、ベクトル空間内で質問自体に近い位置にあるということです。

あとは、最も近い情報ベクトルに対応するテキストをプロンプトに組み込み、そのプロンプトを LLM に送信してその情報をもとに質問に回答させる関数を作成するだけです。

まず、このプロンプトを組み立てるヘルパー関数の作成から始めます。`context` は（最近傍探索によって見つかった）関連情報の文字列となり、`query` はクエリ文字列となります。


### 演習
下の関数を完成させてテンプレート化されたプロンプトを実装し、クエリ文字列と関連するコンテキスト文字列を結合するようにしてください。


In [ ]:
def get_prompt(query: str, context: list[str]):
    # TODO: プロンプト テンプレートを作成
    prompt = None
    return prompt


これらをすべて以下の処理を行う 1 つの関数にまとめます:
* クエリを埋め込む
* クエリ ベクトルとすべての情報ベクトルとの間の距離を計算する
* 距離で並べ替えて最も関連性の高い k 個の情報テキストを取得する
* 最も関連性の高い k 個の情報テキストをクエリとともに LLM へのプロンプトで使用する
* LLM の応答と使用された情報（引用）を返す


### 演習
エンドツーエンドの検索チェーンを実装するように、下の関数を完成させてください。


In [ ]:
def retrieval_chain(query: str, k: int = 2):
    # TODO: クエリとすべての情報ベクトルとの距離を計算
    distances = None

    # TODO: 情報を距離が小さい順（最も近い順）に並べ替え
    sorted_distances = None

    # TODO: 最も近い k 個のベクトルに対応するテキストを取得
    closest_information_texts = None

    # TODO: 最も近い k 個の情報テキストを LLM へのプロンプトに組み込む
    prompt = None

    # TODO: プロンプトを LLM に送信
    response = None
    print(f"Response: {response.text}")
    print(f"Information used: {closest_information_texts}")


In [ ]:
retrieval_chain("What type of animal is Estrella?")

In [ ]:
retrieval_chain("How many times a day do I need to feed Finnegan?")

In [ ]:
retrieval_chain("What stock should I invest in this month?")

提供された情報から回答できない質問がされた場合、LLM が無理に答えようとしないようにプロンプトが構築されていることに注目してください。

また、ここでは任意に k=2（最も近い 2 つの情報テキストをプロンプトに含める）を設定していることにも注意が必要です。ユースケースによって必要な k は異なり、万能な唯一の正解というものはありません。


### LangChain と Chroma による簡素化とスケーリング
もちろん、グラウンディング情報が 5 つの例だけであれば、5 つすべてをプロンプトに簡単に含めることができます。言い換えれば、プロンプトに*何*が必要かを特定するための追加の検索ステップは不要でした。しかし現実世界では、数千から数百万のグラウンディング情報の例が存在する可能性があります。さらに、グラウンディングの例の数が増えるにつれて、すべての単一ベクトルに対して距離を計算するだけでは非常に非効率になります。つまり、本番環境の検索拡張生成システムには以下が必要です:
* 大量の情報を保存するためのスケーラブルなベクトル データベース
* 最近傍探索を効率的に実行する方法

もちろん、ベクトル ストアには [Vector Search](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/overview) のようなマネージドでスケーラブルなサービスを含め、多くの選択肢があります。このラボではわかりやすさを重視し、ベクトル ストアとして [Chroma](https://python.langchain.com/docs/integrations/vectorstores/chroma) を、検索システムのオーケストレーションに [LangChain](https://github.com/langchain-ai/langchain) を使用します。LangChain は、上記の簡単な例で自前で実装する必要があったステップを簡素化するのに役立つクラスとメソッドを提供します。


#### ドキュメントの読み込み

このノートブックでは、Wikipedia API を使用して、大規模言語モデルに関する Wikipedia 記事の独自のナレッジベースを作成します。


In [ ]:
def search_wikipedia(query, language='en', limit=5):
    wiki_wiki = wikipediaapi.Wikipedia(user_agent='https://cloud.google.com/asl', language=language)
    docs = []
    results = wiki_wiki.search(query, limit=limit)
    for title, page in results.pages.items():
        doc = Document(
            page_content=page.text,
            metadata={
                'title': title,
                'source': page.canonicalurl
            }
        )
        docs.append(doc)
    return docs

docs = search_wikipedia("Large Language Models")

# 単一のドキュメントを確認
docs[0]


#### テキストのチャンクへの分割
ドキュメントを取得したので、それらをチャンク（塊）に分割します。各チャンクがベクトル ストア内の 1 つのベクトルになります。これを行うために、チャンク サイズ（文字数）とチャンク オーバーラップ（重複量、つまりスライディング ウィンドウ）を定義します。最適なチャンク サイズを決定するのは難しい場合があります。チャンク サイズが大きすぎると、チャンクあたりの情報が多すぎ（個々のチャンクが具体的でなくなり）、逆に小さすぎるとチャンクあたりの情報が不足します。どちらの場合も、クエリ/質問の埋め込みを使用した最近傍探索で実際に関連するチャンクを取得するのに苦労したり、チャンクが大きすぎて LLM クエリのコンテキストとして使用できなくなったりする可能性があります。

このノートブックでは 800 文字のチャンク サイズと 400 文字のチャンク オーバーラップを使用しますが、他のサイズも自由に試してみてください！注: Python の len 関数以外でチャンク サイズ/オーバーラップを決定したい場合は、`RecursiveCharacterTextSplitter` でカスタムの `length_function` を指定できます。`RecursiveCharacterTextSplitter` 以外にも、検討可能な [他のテキスト スプリッター](https://python.langchain.com/docs/how_to/#text-splitters) があります。


### 演習
下のセルで、ドキュメントを分割・チャンク化するために `RecursiveCharacterTextSplitter` をインスタンス化してください。


In [ ]:
text_splitter = None  # TODO: RecursiveCharacterTextSplitter をインスタンス化

chunks = text_splitter.split_documents(docs)

# 最初の 2 つのチャンクを確認
chunks[0:2]


In [ ]:
print(f"Number of documents: {len(docs)}")
print(f"Number of chunks: {len(chunks)}")

#### ドキュメント チャンクの埋め込み
次に、ドキュメント チャンクを埋め込んでベクトル ストアに保存する必要があります。これには任意のテキスト埋め込みモデルを使用できますが、予測時にクエリ/質問を埋め込む際にも同じテキスト埋め込みモデルを必ず使用する必要があります。ここではシンプルにするために、Gemini API for Embeddings を使用します。LangChain ライブラリは、Gemini Embeddings API の使いやすいラッパー クラスである `GoogleGenerativeAIEmbeddings` を提供しています。

Vector Search はインデックスの作成に時間がかかる（約 45 分）ため、シンプルにするために代わりに Chroma を使用します。もちろん、大規模な独自のナレッジベースを持つ実際のユースケースでは、すべてをメモリに収めることができない場合があります。LangChain には Chroma 用の優れたラッパー クラスがあり、ドキュメントのリストと埋め込みクラスを渡すことでベクトル ストアを作成できます。


In [ ]:
embedding = GoogleGenerativeAIEmbeddings(vertexai=True, model=EMBEDDING_MODEL)

# ベクトル ストアがディスクに保存されるように persist_directory を設定
db = Chroma.from_documents(chunks, embedding, persist_directory="./vectorstore")


#### すべての統合

すべての準備が整ったので、LangChain のチェーンを使用してこれらすべてを連携させます。LangChain チェーンは、特定のユースケースで LLM を使用するために必要な複数のステップをオーケストレーション（統括）するだけです。この場合、連携させるプロセスは、まずクエリ/質問を埋め込み、次に最近傍探索を実行して関連するチャンクを見つけ、関連するチャンクを使用して LLM で応答を作成します。ベクトル ストアとして Chroma データベースを、LLM として Gemini を使用します。LangChain は Gemini のラッパーである `GoogleGenerativeAI` を提供しています。

このシンプルな Q&A のユースケースでは、LangChain の `RetrievalQA` を使用してプロセスを連結できます。


### 演習
下のセルで、LangChain の `RetrievalQA` クラスを使用してエンドツーエンドの検索チェーンを実装してください。


In [ ]:
# ベクトル ストア
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10},  # 取得する最近傍の数
)

# temperature、top_p、top_k も設定可能
llm = GoogleGenerativeAI(model=GENERATIVE_MODEL, max_output_tokens=1024, vertexai=True)

# Q&A チェーン
qa = None  # TODO: 検索チェーンをインスタンス化


すべてが連携されたので、クエリを送信して回答を得ることができます！


In [ ]:
def ask_question(question: str):
    response = qa.invoke({"query": question})
    print(f"Response: {response['result']}\n")

    citations = {doc.metadata["source"] for doc in response["source_documents"]}
    print(f"Citations: {citations}\n")

    # 使用されたソース チャンクを出力する場合は以下をコメント解除
    # print(f"Source Chunks Used: {response['source_documents']}")


In [ ]:
ask_question("What technology underpins large language models?")

In [ ]:
ask_question("When was the transformer introduced?")

お疲れ様でした！これで、簡単な検索拡張生成システムをゼロから構築し、その学びを応用してベクトル データベースと LangChain によるオーケストレーションを使用したより実践的なシステムを構築することができました。
